In [ ]:
"""
Pre-final-test checks:
1. Confirm no overfitting (train vs val metrics)
2. Tune the classification threshold on validation data (NOT test data)
   to prioritize recall - fraud detection favors catching fraud over
   avoiding false alarms, since missed fraud costs more than a manual review.
"""

import pandas as pd
import joblib
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)


# 1. Load data + model

X_train = pd.read_csv("../dataset/processed/X_train.csv")
y_train = pd.read_csv("../dataset/processed/y_train.csv").values.ravel()
X_val = pd.read_csv("../dataset/processed/X_val.csv")
y_val = pd.read_csv("../dataset/processed/y_val.csv").values.ravel()

model = joblib.load("../models/xgboost_model.joblib")


# 2. Overfitting check: compare train vs val AUC

train_proba = model.predict_proba(X_train)[:, 1]
val_proba = model.predict_proba(X_val)[:, 1]

train_auc = roc_auc_score(y_train, train_proba)
val_auc = roc_auc_score(y_val, val_proba)

print("=" * 60)
print("OVERFITTING CHECK")
print("=" * 60)
print(f"Train AUC-ROC: {train_auc:.4f}")
print(f"Val AUC-ROC:   {val_auc:.4f}")
print(f"Gap:           {train_auc - val_auc:.4f}")
print("(A small gap like <0.01 is healthy. A large gap means overfitting.)")


# 3. Threshold tuning on validation data only

print("\n" + "=" * 60)
print("THRESHOLD TUNING (on validation set)")
print("=" * 60)

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
results = []

for t in thresholds:
    y_pred_t = (val_proba >= t).astype(int)
    results.append({
        "threshold": t,
        "precision": precision_score(y_val, y_pred_t, zero_division=0),
        "recall": recall_score(y_val, y_pred_t, zero_division=0),
        "f1": f1_score(y_val, y_pred_t, zero_division=0),
        "accuracy": accuracy_score(y_val, y_pred_t),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

print("\nPick the threshold that best matches your priority:")
print("- Lower threshold -> higher recall (catches more fraud, more false alarms)")
print("- Higher threshold -> higher precision (fewer false alarms, may miss some fraud)")
print("\nSave your chosen threshold - you'll hardcode it into the backend's prediction logic.")

OVERFITTING CHECK
Train AUC-ROC: 1.0000
Val AUC-ROC:   0.9998
Gap:           0.0002
(A small gap like <0.01 is healthy. A large gap means overfitting.)

THRESHOLD TUNING (on validation set)
 threshold  precision   recall       f1  accuracy
       0.1   0.814861 0.997726 0.897069  0.999703
       0.2   0.897069 0.997726 0.944724  0.999849
       0.3   0.917713 0.997726 0.956048  0.999881
       0.4   0.928723 0.997726 0.961988  0.999898
       0.5   0.943369 0.997726 0.969786  0.999919
       0.6   0.950867 0.997726 0.973733  0.999930
       0.7   0.957091 0.997726 0.976986  0.999939

Pick the threshold that best matches your priority:
- Lower threshold -> higher recall (catches more fraud, more false alarms)
- Higher threshold -> higher precision (fewer false alarms, may miss some fraud)

Save your chosen threshold - you'll hardcode it into the backend's prediction logic.
